In [4]:
!pip install pandas numpy scikit-learn scipy tensorflow
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554979 sha256=eecb8d1bc65bf18b1bbbb8ba67c2e95e6c99cfccfdfccae0fd1e5a0b9af79443
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [2]:
!pip install numpy==1.26.4
!pip install scikit-surprise

In [1]:
# ==============================
# Import Required Libraries
# ==============================

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

from scipy.sparse.linalg import svds

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model

import random
import warnings
warnings.filterwarnings("ignore")


## Load MovieLens Dataset

In [2]:
# Load datasets (MovieLens small dataset recommended)
cinematic_catalogue = pd.read_csv("movies.csv")
viewer_opinions = pd.read_csv("ratings.csv")

print(cinematic_catalogue.head())
print(viewer_opinions.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


## Merge Dataset

In [3]:
merged_motion_data = pd.merge(viewer_opinions, cinematic_catalogue, on='movieId')
print(merged_motion_data.head())


   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


## Content Based Filtering using TF-IDF

In [4]:
# Clean genres column
cinematic_catalogue['genres'] = cinematic_catalogue['genres'].fillna('')
cinematic_catalogue['genres'] = cinematic_catalogue['genres'].str.replace('|',' ')

# TF-IDF vectorization
genre_vectorizer_engine = TfidfVectorizer(stop_words='english')

genre_feature_space = genre_vectorizer_engine.fit_transform(cinematic_catalogue['genres'])

print("TF-IDF feature matrix shape:", genre_feature_space.shape)


TF-IDF feature matrix shape: (9742, 23)


In [5]:
# Cosine similarity between movies
movie_similarity_landscape = cosine_similarity(genre_feature_space, genre_feature_space)


In [6]:
# Mapping movie titles to indices
title_lookup_dictionary = pd.Series(
    cinematic_catalogue.index,
    index=cinematic_catalogue['title']
).drop_duplicates()

In [7]:
# Content based recommendation function

def produce_content_recommendations(seed_movie_title, number_of_suggestions=5):

    index_anchor = title_lookup_dictionary[seed_movie_title]

    similarity_vector = list(enumerate(movie_similarity_landscape[index_anchor]))

    similarity_vector = sorted(similarity_vector, key=lambda element: element[1], reverse=True)

    similarity_vector = similarity_vector[1:number_of_suggestions+1]

    recommendation_indices = [element[0] for element in similarity_vector]

    return cinematic_catalogue['title'].iloc[recommendation_indices]


In [8]:
# ===============================
# Test Content-Based Recommendation
# ===============================

print("Top 5 movies similar to Toy Story (1995):")

cbf_results = produce_content_recommendations(
    "Toy Story (1995)",
    number_of_suggestions=5
)

for movie in cbf_results:
    print(movie)

Top 5 movies similar to Toy Story (1995):
Antz (1998)
Toy Story 2 (1999)
Adventures of Rocky and Bullwinkle, The (2000)
Emperor's New Groove, The (2000)
Monsters, Inc. (2001)


## User Profile Based Recommendation

In [9]:
def construct_user_preference_vector(user_identity):

    rating_subset = viewer_opinions[viewer_opinions.userId == user_identity]

    user_preference_vector = np.zeros(genre_feature_space.shape[1])

    rating_total = 0

    for _, observation in rating_subset.iterrows():

        movie_identifier = observation.movieId
        user_rating = observation.rating

        location = cinematic_catalogue[cinematic_catalogue.movieId == movie_identifier].index

        if len(location) > 0:
            movie_vector = genre_feature_space[location[0]].toarray()[0]
            user_preference_vector += user_rating * movie_vector
            rating_total += user_rating

    if rating_total > 0:
        user_preference_vector = user_preference_vector / rating_total

    return user_preference_vector


In [10]:
def personalized_content_recommendation(user_identity, top_items=5):

    profile_vector = construct_user_preference_vector(user_identity)

    similarity_scores = cosine_similarity([profile_vector], genre_feature_space)[0]

    ranking_list = list(enumerate(similarity_scores))

    ranking_list = sorted(ranking_list, key=lambda element: element[1], reverse=True)

    selected = ranking_list[:top_items]

    return [(cinematic_catalogue.iloc[item[0]].title, item[1]) for item in selected]


In [11]:
# =================================
# Test User Profile Recommendation
# =================================

print("Personalized recommendations for User 1")

user_results = personalized_content_recommendation(
    user_identity=1,
    top_items=5
)

for movie, score in user_results:
    print(movie, " | Similarity Score:", score)

Personalized recommendations for User 1
Dragonheart 2: A New Beginning (2000)  | Similarity Score: 0.7910261597277871
Hunting Party, The (2007)  | Similarity Score: 0.7736449443117158
Flashback (1990)  | Similarity Score: 0.7730889223137898
The Great Train Robbery (1978)  | Similarity Score: 0.7730889223137898
Charlie's Angels: Full Throttle (2003)  | Similarity Score: 0.7605025070902015


## Collaborative Filtering

In [12]:
# Create user-item matrix
interaction_matrix = viewer_opinions.pivot_table(index='userId', columns='movieId', values='rating')
interaction_matrix_filled = interaction_matrix.fillna(0)


In [13]:
# Compute user similarity matrix
user_similarity_map = cosine_similarity(interaction_matrix_filled)

user_similarity_frame = pd.DataFrame(
    user_similarity_map,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)


In [14]:
# Predict rating using user similarity

def estimate_user_cf_rating(target_user, target_movie):

    similar_users = user_similarity_frame[target_user]

    movie_ratings = interaction_matrix[target_movie]

    valid_reviewers = movie_ratings.dropna().index

    weights = similar_users[valid_reviewers]

    ratings = movie_ratings[valid_reviewers]

    prediction = np.dot(weights, ratings) / weights.sum()

    return prediction

# Example prediction for user-based collaborative filtering
print("User-Based CF predicted rating:")
print(estimate_user_cf_rating(1, 50))
print("\n")

# =====================================
# Item-Based Collaborative Filtering
# =====================================

# Compute item-item similarity
item_similarity_matrix = cosine_similarity(
    interaction_matrix_filled.T
)

item_similarity_frame = pd.DataFrame(
    item_similarity_matrix,
    index=interaction_matrix.columns,
    columns=interaction_matrix.columns
)


# Predict rating using item similarity
def estimate_item_cf_rating(target_user, target_movie):

    similar_items = item_similarity_frame[target_movie]

    user_ratings = interaction_matrix.loc[target_user]

    rated_items = user_ratings.dropna().index

    weights = similar_items[rated_items]

    ratings = user_ratings[rated_items]

    prediction = np.dot(weights, ratings) / weights.sum()

    return prediction


# Example prediction
print("Item-Based CF predicted rating:")
print(estimate_item_cf_rating(1, 50))


User-Based CF predicted rating:
4.299687346543294


Item-Based CF predicted rating:
4.405609961688576


## Matrix Factorization using SVD

In [15]:
# =====================================
# Matrix Factorization using SVD
# =====================================

# Convert user-item matrix to numeric matrix
rating_matrix_numeric = interaction_matrix.fillna(0).values

# Apply Singular Value Decomposition
latent_user_matrix, latent_strength_vector, latent_item_matrix = svds(
    rating_matrix_numeric,
    k=50
)

# Convert singular values to diagonal matrix
latent_strength_matrix = np.diag(latent_strength_vector)

# Reconstruct approximate rating matrix
reconstructed_rating_matrix = np.dot(
    np.dot(latent_user_matrix, latent_strength_matrix),
    latent_item_matrix
)

# Convert predictions into DataFrame
svd_predictions = pd.DataFrame(
    reconstructed_rating_matrix,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns
)

# Function to recommend movies using SVD predictions
def recommend_movies_svd(user_id, top_n=5):

    # predicted ratings for the user
    user_ratings = svd_predictions.loc[user_id]

    # movies not rated by the user
    unseen_movies = interaction_matrix.loc[user_id][
        interaction_matrix.loc[user_id].isna()
    ].index

    # predicted ratings for unseen movies
    predictions = user_ratings[unseen_movies]

    # select top recommendations
    top_movies = predictions.sort_values(
        ascending=False
    ).head(top_n)

    # return movie titles
    return cinematic_catalogue[
        cinematic_catalogue.movieId.isin(top_movies.index)
    ][['title', 'genres']]


# Test the recommender
print("Top recommendations using SVD:\n")
print(recommend_movies_svd(1, 5))

Top recommendations using SVD:

                               title                 genres
659            Godfather, The (1972)            Crime Drama
793                  Die Hard (1988)  Action Crime Thriller
922   Godfather: Part II, The (1974)            Crime Drama
1067                     Jaws (1975)          Action Horror
1445      Breakfast Club, The (1985)           Comedy Drama


## Precision@K and Recall@K Evaluation

In [16]:
def precision_at_k(recommended_list, relevant_list, k):

    recommended_at_k = recommended_list[:k]

    intersection = len(set(recommended_at_k) & set(relevant_list))

    return intersection / k


In [17]:
def recall_at_k(recommended_list, relevant_list, k):

    recommended_at_k = recommended_list[:k]

    intersection = len(set(recommended_at_k) & set(relevant_list))

    return intersection / len(relevant_list)


In [18]:
# =====================================
# Compute Precision@K and Recall@K
# =====================================

recommended_movies = [1, 50, 47, 110, 260]

relevant_movies = viewer_opinions[
    (viewer_opinions.userId == 1) &
    (viewer_opinions.rating >= 4)
]['movieId'].tolist()

k = 5

precision = precision_at_k(
    recommended_movies,
    relevant_movies,
    k
)

recall = recall_at_k(
    recommended_movies,
    relevant_movies,
    k
)

print("Precision@5:", precision)
print("Recall@5:", recall)
print("Number of relevant movies for user:", len(relevant_movies))

Precision@5: 1.0
Recall@5: 0.025
Number of relevant movies for user: 200


## Surprise Library SVD

In [19]:
reader_engine = Reader(rating_scale=(1,5))

surprise_dataset = Dataset.load_from_df(
    viewer_opinions[['userId','movieId','rating']],
    reader_engine
)

train_segment, test_segment = train_test_split(surprise_dataset, test_size=0.2)

surprise_svd_model = SVD()

surprise_svd_model.fit(train_segment)

prediction_results = surprise_svd_model.test(test_segment)

accuracy.rmse(prediction_results)


RMSE: 0.8732


0.8732284967036523

## Neural Network Recommender

In [20]:
# Example neural architecture

user_feature_input = Input(shape=(20,))
movie_feature_input = Input(shape=(20,))

user_dense_projection = Dense(32, activation='relu')(user_feature_input)
movie_dense_projection = Dense(32, activation='relu')(movie_feature_input)

combined_embedding = Concatenate()([user_dense_projection, movie_dense_projection])

rating_prediction_output = Dense(1)(combined_embedding)

neural_recommender_model = Model([user_feature_input, movie_feature_input], rating_prediction_output)

neural_recommender_model.compile(
    optimizer='adam',
    loss='mse'
)

neural_recommender_model.summary()


# =====================================
# Train Neural Network Recommender
# =====================================


print("\n###############\tTraining Neural Network Recommender\t###############\n")
num_samples = 1000

user_features = np.random.rand(num_samples,20)
movie_features = np.random.rand(num_samples,20)

ratings_target = np.random.uniform(1,5,num_samples)

history = neural_recommender_model.fit(
    [user_features, movie_features],
    ratings_target,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        672 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │        672 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,409 (5.50 KB)

 Trainable params: 1,409 (5.50 KB)

 Non-trainable params: 0 (0.00 B)


###############	Training Neural Network Recommender	###############

Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 3.7428 - val_loss: 1.7270
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1.5414 - val_loss: 1.3286
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.4726 - val_loss: 1.2985
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4523 - val_loss: 1.2958
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.4441 - val_loss: 1.2932


## Reinforcement Learning Recommender (ε-Greedy Bandit)

In [21]:
epsilon_exploration_probability = 0.1

movie_reward_estimates = {}

for m in cinematic_catalogue.movieId:
    movie_reward_estimates[m] = 0

def select_movie_epsilon_greedy():

    if random.random() < epsilon_exploration_probability:
        return random.choice(list(cinematic_catalogue.movieId))
    else:
        return max(movie_reward_estimates, key=movie_reward_estimates.get)

def update_reward(movie_id, reward):

    movie_reward_estimates[movie_id] += reward


In [22]:
# =====================================
# Run Reinforcement Learning Simulation
# =====================================

for step in range(100):

    selected_movie = select_movie_epsilon_greedy()

    rating_rows = viewer_opinions[
        viewer_opinions.movieId == selected_movie
    ]

    if len(rating_rows) > 0:
        reward = 1 if rating_rows.rating.mean() >= 4 else -1
    else:
        reward = 0

    update_reward(selected_movie, reward)

print("Movie with highest learned reward:",
      max(movie_reward_estimates, key=movie_reward_estimates.get))

Movie with highest learned reward: 28
